#### SAMPLE DATASET (RAW - BRONZE INPUT)


In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

spark = SparkSession.builder.getOrCreate()

data = [
    (101, "C001", "Laptop", "Electronics", "Hyderabad", "2024-01-01", "50000", 1),
    (102, "C002", "Mobile", "Electronics", "Chennai", "2024-01-02", None, 2),
    (103, "C001", "Tablet", "Electronics", "Hyderabad", "2024-01-03", "20000", 1),
    (104, "C003", "Laptop", "Electronics", "Delhi", "2024-01-04", "55000", 1),
    (105, "C002", "Mobile", "Electronics", "Chennai", "2024-01-05", "18000", 1),
    (106, "C004", "Watch", "Accessories", "Mumbai", "2024-01-06", "8000", 1),
    (103, "C001", "Tablet", "Electronics", "Hyderabad", "2024-01-07", "22000", 1),
    (107, "C005", "Headphones", "Accessories", None, "2024-01-08", "3000", 1),
    (108, "C006", "Laptop", "Electronics", "Bangalore", "2024-01-09", "-45000", 1),
    (109, "C007", "Mobile", "Electronics", "Chennai", "2024-01-10", "15000", 2),
    (109, "C007", "Mobile", "Electronics", "Chennai", "2024-01-10", "15000", 2)
]

columns = ["order_id", "customer_id", "product", "category", "city", "date", "amount", "quantity"]

df = spark.createDataFrame(data, columns)

# Write to Bronze
df.write.format("delta") \
  .mode("append") \
  .saveAsTable("medallion_architecture.bronze.flipkart_orders")

# silver Layer


##### 
Data Cleaning
Handle NULL values (amount, city)
Decide: fill or remove

Data Type Fix
Convert amount → integer
Convert date → proper date format

Remove Duplicates
Identify duplicate order_id
Keep only one record

Handle Updates
Same order_id appears multiple times

Keep latest record based on date
Data Validation

Remove negative amounts
Ensure valid data
Standardization

Format city names (optional)

#### Handling null values 

In [0]:
df=df.withColumn("amount",when(col("amount").isNull(),"0").otherwise(col("amount")))
df=df.fillna({"city":"unknown"})
display(df)

##### removing duplicates

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, col

# Create window partition by order_id
window = Window.partitionBy("order_id").orderBy(col("date").desc())

# Assign row number
df = df.withColumn("rn", row_number().over(window))

# Keep only latest record
df = df.filter(col("rn") == 1).drop("rn")
display(df)

#### Type casting

In [0]:
# 2. Type casting
df = df.withColumn("amount", col("amount").cast("int")) \
       .withColumn("date", to_date(col("date")))
display(df)

#### Remove negative sales

In [0]:

df = df.filter(col("amount") > 0)
display(df)

####Write to Silver

In [0]:

df.write.format("delta") \
  .mode("overwrite") \
  .saveAsTable("medallion_architecture.silver.flipkart_orders_cleaned")

# Gold Layer

In [0]:
df = spark.read.table("medallion_architecture.silver.flipkart_orders_cleaned")

In [0]:

df.write.format("delta") \
  .mode("overwrite") \
  .saveAsTable("medallion_architecture.gold.flipkart_orders_cleaned")

In [0]:
# total sales by product
display(df)